In [ ]:
import torch, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from scipy.stats import spearmanr

import os
import json

LAYER = 16 
POOL  = "mean" 

def find_local_model_path(model_name):
    """
    Use HF model id to search model path from local cache.
    """
    cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
    model_name_safe = model_name.replace("/", "--")
    model_cache = os.path.join(cache_dir, f"models--{model_name_safe}")
    
    if not os.path.exists(model_cache):
        print(f"Model cache not found at {model_cache}")
        return None
    
    # search snapshots dir
    snapshots_dir = os.path.join(model_cache, "snapshots")
    if os.path.exists(snapshots_dir):
        # get lateset snapshot
        snapshots = os.listdir(snapshots_dir)
        if snapshots:
            latest_snapshot = sorted(snapshots)[-1]
            model_path = os.path.join(snapshots_dir, latest_snapshot)
            print(f"Found model at: {model_path}")
            
            # find config.json
            config_path = os.path.join(model_path, "config.json")
            if os.path.exists(config_path):
                print(f"✓ config.json found")
                return model_path
            else:
                print(f"✗ config.json not found in {model_path}")
    
    return None

def load_model(ckpt_path):
    tok = AutoTokenizer.from_pretrained(ckpt_path)
    model = AutoModelForCausalLM.from_pretrained(
        ckpt_path, torch_dtype=torch.float16,
        output_hidden_states=True)
    
    #os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE
    model.to(torch.device(f"cuda:0"))
    model.eval()
    return model, tok

@torch.no_grad()
def get_repr(text, model, tok, layer=LAYER, pool=POOL, max_len=1024):
    ids = tok(text, return_tensors="pt", truncation=True,
              max_length=max_len).input_ids.to(model.device)
    out = model(ids)                       # hidden_states: tuple(L+1) each [1,T,H]
    h = out.hidden_states[layer][0] 
    if pool == "mean":
        vec = h.mean(dim=0)
    else:  # last token
        vec = h[-1]
    return vec.float().cpu().numpy() 

def build_features(data, model, tok):
    # data: {concept: {"safe":[...], "unsafe":[...]}}
    X, y, concepts = [], [], []
    for concept, v in data.items():
        for txt in v["safe"]:
            X.append(get_repr(txt, model, tok)); y.append(0); concepts.append(concept)
        for txt in v["unsafe"]:
            X.append(get_repr(txt, model, tok)); y.append(1); concepts.append(concept)
    return np.array(X), np.array(y), np.array(concepts)

def probe_separability(X, y, seed=0):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, C=1.0)
    )
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    acc = cross_val_score(clf, X, y, cv=skf, scoring="accuracy")
    auc = cross_val_score(clf, X, y, cv=skf, scoring="roc_auc")
    return acc.mean(), acc.std(), auc.mean()

def load_CKPS(ckps_path):
    model_to_eval = [os.path.join(ckps_path, file) for file in os.listdir(ckps_path) if 'checkpoint-' in file]
    model_to_eval = sorted(model_to_eval, key=lambda x: int(x.split('-')[-1]))
    CKPTS = {f"E{i+1}": path for i, path in enumerate(model_to_eval)}
    return CKPTS

def print_separability(ckps_path):
    print(f"Evaluating separability for checkpoints in {ckps_path}")
    CKPTS = load_CKPS(ckps_path)

    results = {}
    results["E0"] = {"acc": 0.838, "sd": 0.018, "auc": 0.906}
    print("E0: probe acc=0.838±0.018, auc=0.906")

    for name, path in CKPTS.items():
        model, tok = load_model(path)
        X, y, concepts = build_features(data, model, tok)
        acc, sd, auc = probe_separability(X, y)
        print(f"{name}: probe acc={acc:.3f}±{sd:.3f}, auc={auc:.3f}")
        results[name] = {"acc": acc, "sd": sd, "auc": auc}
        
        model.to("cpu")
        del model
        torch.cuda.empty_cache()
    return results

os.environ["CUDA_VISIBLE_DEVICES"] = "6"
model_name = "meta-llama/Llama-3.1-8B-Instruct"
path = find_local_model_path(model_name)

with open("test_dataset_plain.json", "r") as f:
    data = json.load(f)

Found model at: /home/sxw/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659
✓ config.json found


In [ ]:
def load_CKPS(ckps_path):
    model_to_eval = [os.path.join(ckps_path, file) for file in os.listdir(ckps_path) if 'checkpoint-' in file]
    model_to_eval = sorted(model_to_eval, key=lambda x: int(x.split('-')[-1]))
    CKPTS = {f"E{i+1}": path for i, path in enumerate(model_to_eval)}
    return CKPTS

def print_separability(ckps_path):
    print(f"Evaluating separability for checkpoints in {ckps_path}")
    CKPTS = load_CKPS(ckps_path)

    results = {}
    results["E0"] = {"acc": 0.838, "sd": 0.018, "auc": 0.906}
    print("E0: probe acc=0.838±0.018, auc=0.906")

    for name, path in CKPTS.items():
        model, tok = load_model(path)
        X, y, concepts = build_features(data, model, tok)
        acc, sd, auc = probe_separability(X, y)
        print(f"{name}: probe acc={acc:.3f}±{sd:.3f}, auc={auc:.3f}")
        results[name] = {"acc": acc, "sd": sd, "auc": auc}
        
        model.to("cpu")
        del model
        torch.cuda.empty_cache()
    return results

#ckps_path = "/FTLoss/llama-beavertails-Random_n2000/model_ckps"
#ckps_path = "/FTLoss/llama-beavertails-SafeOnlyBalanced_perconcept20/model_ckps"
ckps_path = "/FTLoss/llama-beavertails-UnsafeOnlyBalanced_perconcept20/model_ckps"
results = print_separability(ckps_path)

Evaluating separability for checkpoints in /home/sxw/FTLoss/llama-beavertails-UnsafeOnlyBalanced_perconcept20/model_ckps
E0: probe acc=0.838±0.018, auc=0.906


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

E1: probe acc=0.837±0.018, auc=0.901


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

E3: probe acc=0.842±0.022, auc=0.902


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

E4: probe acc=0.843±0.021, auc=0.903


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

E5: probe acc=0.841±0.019, auc=0.902


In [ ]:
Random_n2000
E0: probe acc=0.838±0.018, auc=0.906
E1: probe acc=0.842±0.018, auc=0.902
E2: probe acc=0.838±0.017, auc=0.904
E3: probe acc=0.839±0.017, auc=0.903
E4: probe acc=0.843±0.015, auc=0.905
E5: probe acc=0.841±0.014, auc=0.903

SafeOnlyBalanced_perconcept20
E0: probe acc=0.838±0.018, auc=0.906
E1: probe acc=0.838±0.022, auc=0.904
E2: probe acc=0.837±0.019, auc=0.902
E3: probe acc=0.843±0.018, auc=0.905
E4: probe acc=0.847±0.015, auc=0.904
E5: probe acc=0.843±0.014, auc=0.904

UnsafeOnlyBalanced_perconcept20
E0: probe acc=0.838±0.018, auc=0.906
E1: probe acc=0.837±0.018, auc=0.901
E2: probe acc=0.841±0.021, auc=0.901
E3: probe acc=0.842±0.022, auc=0.902
E4: probe acc=0.843±0.021, auc=0.903
E5: probe acc=0.841±0.019, auc=0.902

In [ ]:
import os, json, numpy as np, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.stats import spearmanr, pearsonr

LAYER = 16
POOL  = "mean"

def concept_ppl_separation(ppl_epoch):
    """
    per-concept PPL separation
    ppl_epoch: {concept: {'safe':[...], 'unsafe':[...]}}  
    returns {concept: mean(PPL_unsafe) - mean(PPL_safe)}
    """
    sep = {}
    for c, v in ppl_epoch.items():
        sep[c] = float(np.mean(v['unsafe']) - np.mean(v['safe']))
    return sep

@torch.no_grad()
def get_repr(text, model, tok, layer=LAYER, pool=POOL, max_len=1024):
    """per-concept probe separability"""
    ids = tok(text, return_tensors="pt", truncation=True,
              max_length=max_len).input_ids.to(model.device)
    h = model(ids).hidden_states[layer][0]        # [T,H]
    vec = h.mean(0) if pool=="mean" else h[-1]
    return vec.float().cpu().numpy()

def build_features(data, model, tok):
    X, y, concepts = [], [], []
    for c, v in data.items():
        for txt in v['safe']:
            X.append(get_repr(txt, model, tok)); y.append(0); concepts.append(c)
        for txt in v['unsafe']:
            X.append(get_repr(txt, model, tok)); y.append(1); concepts.append(c)
    return np.array(X), np.array(y), np.array(concepts)

def concept_probe_margin(X, y, concepts):
    """Train linear probe 
       per-concept signed margin = mean(margin of unsafe) - mean(margin of safe).
    """
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=2000, C=1.0)).fit(X, y)
    margin = clf.decision_function(X)          # signed, + toward unsafe
    out = {}
    for c in np.unique(concepts):
        m = margin[concepts==c]
        yc = y[concepts==c]
        out[c] = float(m[yc==1].mean() - m[yc==0].mean())   # signed separability
    return out

def correlate(probe_sep, ppl_sep):
    common = [c for c in probe_sep if c in ppl_sep]
    a = [probe_sep[c] for c in common]
    b = [ppl_sep[c]   for c in common]
    rho, p_s = spearmanr(a, b)
    r,   p_p = pearsonr(a, b)
    return rho, p_s, r, p_p, common



In [2]:
def load_CKPS(ckps_path):
    model_to_eval = [os.path.join(ckps_path, file) for file in os.listdir(ckps_path) if 'checkpoint-' in file]
    model_to_eval = sorted(model_to_eval, key=lambda x: int(x.split('-')[-1]))
    return model_to_eval

os.environ["CUDA_VISIBLE_DEVICES"] = "5"

data = json.load(open("test_dataset_plain.json"))

strategy = "Random_n2000"
print(f"Evaluating strategy: {strategy}")

ppl_e0   = json.load(open("/home/sxw/FTLoss/llama-beavertails-BaseModel/eval_ppl/ppl_result_list.json"))
ppl_rest = json.load(open(f"/home/sxw/FTLoss/llama-beavertails-{strategy}/eval_ppl/ppl_result_list.json"))
ppl_list  = ppl_e0 + ppl_rest 
print(len(ppl_list))

# checkpoint paths, aligned to the 5 epochs in ppl_list (CONFIRM ordering!)
# CKPTS = ["path/ckpt_e1","path/ckpt_e2","path/ckpt_e3","path/ckpt_e4","path/ckpt_e5"]
ckps_path = f"/home/sxw/FTLoss/llama-beavertails-{strategy}/model_ckps"
CKPS_e0 = ["/home/sxw/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659"]
CKPS_rest = load_CKPS(ckps_path)
CKPS = CKPS_e0 + CKPS_rest
print(len(CKPS))

for i in range(len(CKPS)):
    name = f"E{i+1}"
    ckps = CKPS[i]
    ppl = ppl_list[i]

    tok = AutoTokenizer.from_pretrained(ckps)
    model = AutoModelForCausalLM.from_pretrained(
        ckps, torch_dtype=torch.float16, output_hidden_states=True).to("cuda:0").eval()

    X, y, concepts = build_features(data, model, tok)
    probe_sep = concept_probe_margin(X, y, concepts)          # 133 signed margins
    ppl_sep   = concept_ppl_separation(ppl)       # 133 separations

    rho, ps, r, pp, _ = correlate(probe_sep, ppl_sep)
    print(f"E{name}: Spearman rho={rho:.3f} (p={ps:.2e}), Pearson r={r:.3f} (p={pp:.2e})")

    model.to("cpu"); del model; torch.cuda.empty_cache()

Evaluating strategy: Random_n2000
6
6


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

EE1: Spearman rho=-0.129 (p=1.38e-01), Pearson r=-0.124 (p=1.54e-01)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

EE4: Spearman rho=-0.124 (p=1.55e-01), Pearson r=-0.145 (p=9.64e-02)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

EE5: Spearman rho=-0.115 (p=1.88e-01), Pearson r=-0.141 (p=1.05e-01)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

EE6: Spearman rho=-0.106 (p=2.25e-01), Pearson r=-0.121 (p=1.64e-01)


In [ ]:
Random_n2000
E0: Spearman rho=-0.129 (p=1.38e-01), Pearson r=-0.124 (p=1.54e-01)
E1: Spearman rho=-0.200 (p=2.10e-02), Pearson r=-0.221 (p=1.07e-02)
E2: Spearman rho=-0.155 (p=7.48e-02), Pearson r=-0.185 (p=3.26e-02)
E3: Spearman rho=-0.124 (p=1.55e-01), Pearson r=-0.145 (p=9.64e-02)
E4: Spearman rho=-0.115 (p=1.88e-01), Pearson r=-0.141 (p=1.05e-01)
E5: Spearman rho=-0.106 (p=2.25e-01), Pearson r=-0.121 (p=1.64e-01)

SafeOnlyBalanced_perconcept20

E0: Spearman rho=-0.129 (p=1.38e-01), Pearson r=-0.124 (p=1.54e-01)
E1: Spearman rho=-0.040 (p=6.46e-01), Pearson r=-0.118 (p=1.78e-01)
E2: Spearman rho=-0.001 (p=9.92e-01), Pearson r=-0.089 (p=3.07e-01)
E3: Spearman rho=0.021 (p=8.11e-01), Pearson r=-0.049 (p=5.76e-01)
E4: Spearman rho=0.078 (p=3.72e-01), Pearson r=0.026 (p=7.70e-01)
E5: Spearman rho=0.098 (p=2.63e-01), Pearson r=0.043 (p=6.24e-01)

UnsafeOnlyBalanced_perconcept20

E0: Spearman rho=-0.129 (p=1.38e-01), Pearson r=-0.124 (p=1.54e-01)
E1: Spearman rho=-0.187 (p=3.10e-02), Pearson r=-0.210 (p=1.55e-02)
E2: Spearman rho=-0.161 (p=6.39e-02), Pearson r=-0.197 (p=2.32e-02)
E3: Spearman rho=-0.137 (p=1.16e-01), Pearson r=-0.167 (p=5.54e-02)
E4: Spearman rho=-0.134 (p=1.23e-01), Pearson r=-0.180 (p=3.80e-02)
E5: Spearman rho=-0.130 (p=1.37e-01), Pearson r=-0.168 (p=5.36e-02)